# 06 ARIMA SARIMA Tuning

Goal: evaluate ARIMA/SARIMA candidate models on validation data and select the champion model.

In [ ]:
from pathlib import Path

from nasdaq_svar.config import load_config
from nasdaq_svar.forecast import _build_candidates, _read_target_series, _score_candidates, _split_series

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "configs").exists() else cwd.parent
cfg = load_config(PROJECT_ROOT / "configs/forecast.yaml")
f_cfg = cfg["forecast"]
series = _read_target_series(PROJECT_ROOT / f_cfg["input_csv"], f_cfg.get("target_column"))
split = _split_series(
    series,
    train_ratio=f_cfg["evaluation"]["train_ratio"],
    valid_ratio=f_cfg["evaluation"]["valid_ratio"],
    min_train_size=f_cfg["evaluation"]["min_train_size"],
)
candidates = _build_candidates(f_cfg["model_space"])
scores = _score_candidates(split, candidates, f_cfg)
scores.head(10)

In [ ]:
best = scores.dropna(subset=["valid_rmse"]).iloc[0]
best

## Notes
- Compare RMSE and MAE first, then AIC/BIC as tie-breakers.
- Keep final model choice fixed before touching test data.